# Siformer VSL100 ? Kaggle GPU from existing landmarks

Upload `dat301m_source_kaggle.zip` and `dat301m_landmark_kaggle.zip` as Kaggle Inputs and enable GPU. The notebook extracts them into `/kaggle/working`, imports the 16-frame landmark archive, runs preflight and smoke, then trains with epoch checkpoints. Set `SOURCE_ROOT` or `LANDMARK_ROOT` in CELL 0 only if discovery is ambiguous. RGB videos are not needed.


In [ ]:
# CELL 0 - TensorFlow settings. Edit here before executing remaining cells. None = unique auto-discovery / preserve local config.
SOURCE_ROOT = None       # Input source folder with train.py, vsl_tf/, vsl/, vendor/, configs/, scripts/...
LANDMARK_ROOT = None     # Input folder containing the recipe directory with index.json and NPZ files
RESTORE_ROOT = None      # Extracted bundle OR old working session with project/run/cache/manifests
WHEELHOUSE = None        # Optional offline wheels folder; otherwise Internet is needed for missing packages
SESSION_NAME = "vsl100_tf_gpu_01"  # Use a NEW name for a new run/resume branch
MODE = "fresh"          # "fresh" or "resume"
RUN_TRAIN = True         # Applies only to fresh mode
RUN_RESUME = False       # OFF by default; explicitly enable for resume
RUN_TEST = False         # OFF by default; enable only after validation-based decisions are final
MAKE_ARCHIVE = True
ADAPTIVE_INFERENCE = False  # Decide before test; False preserves the static baseline
BATCH_SIZE = None        # Preserve local batch size 24; choose a smaller size BEFORE full training if needed
NUM_WORKERS = None       # Preserve local workers=0; optionally choose 2 for a NEW experiment
AMP = None               # Preserve local False; True enables TensorFlow mixed_float16
SEED = None              # Preserve 379
TOTAL_EPOCHS = None      # Preserve 100; resume may increase this, not change other training options
OFFLINE_VARIANTS = None  # Preserve 0: imported landmark archive has no RGB augmentation

In [ ]:
# TensorFlow GPU and environment probe (before any pipeline work).
import os, sys, json, shutil, subprocess, hashlib, platform, csv, zipfile
from pathlib import Path
INPUT, WORKING = Path('/kaggle/input'), Path('/kaggle/working')
if not INPUT.is_dir() or not WORKING.is_dir():
    raise RuntimeError('Run this notebook on Kaggle with Input and Working mounted.')
print('Python', sys.version, 'Platform', platform.platform())
print('Disk', shutil.disk_usage(WORKING))
probe = subprocess.run([sys.executable, '-c',
    "import tensorflow as tf; print('TensorFlow',tf.__version__); "
    "print('Physical GPUs',tf.config.list_physical_devices('GPU')); "
    "print('Logical GPUs',tf.config.list_logical_devices('GPU')); "
    "print('Mixed precision',tf.keras.mixed_precision.global_policy())"],
    text=True, capture_output=True)
print(probe.stdout, probe.stderr)
if probe.returncode or 'Physical GPUs []' in probe.stdout:
    raise RuntimeError('TensorFlow GPU unavailable. Enable GPU Accelerator and select a compatible Kaggle image.')

# Accept the two prepared ZIP files as Kaggle Inputs; extract only into writable Working.
STAGED = WORKING / "dat301m_inputs"
for name in ("dat301m_source_kaggle.zip", "dat301m_landmark_kaggle.zip"):
    matches = list(INPUT.rglob(name))
    if len(matches) > 1:
        raise RuntimeError(f"Multiple {name} files; keep only one Input copy")
    if matches:
        destination = STAGED / Path(name).stem
        if destination.exists():
            raise FileExistsError(f"Extracted Input already exists: {destination}; use a fresh Kaggle session")
        destination.mkdir(parents=True)
        with zipfile.ZipFile(matches[0]) as archive:
            bad = archive.testzip()
            if bad:
                raise ValueError(f"Corrupt ZIP member: {bad}")
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if not target.is_relative_to(destination.resolve()) or member.filename.startswith(("/", "\\")):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
            archive.extractall(destination)
        print("Extracted", matches[0], "to", destination)
SEARCH_ROOTS = (INPUT, STAGED)


In [ ]:
# 3. Discover actual Input paths. Ambiguity is an error: choose explicitly in CELL 0.
REQUIRED = ["train.py", "evaluate.py", "calibrate.py", "prepare_data.py", "requirements.txt",
            "vsl_tf/model.py", "vsl_tf/data.py", "vsl_tf/engine.py", "vsl/adapter.py", "vsl/common.py",
            "vsl/augmentation.py", "vsl/metrics.py", "configs/landmark_train.json", "configs/labels.json", "scripts/import_landmark_cache.py",
            "scripts/preflight.py", "scripts/kaggle_dependencies.py", "scripts/kaggle_bundle.py",
            "docs/dataset_counts.csv", "scripts/model_summary.py",
            "vendor/siformer_official/PROVENANCE.md", "vendor/siformer_official/normalization/__init__.py",
            "vendor/siformer_official/normalization/body_normalization.py",
            "vendor/siformer_official/normalization/hand_normalization.py", "vendor/siformer_official/LICENSE"]
def choose(configured, candidates, name, required):
    candidates = sorted(set(p.resolve() for p in candidates))
    print(name + " candidates:", [str(p) for p in candidates])
    if configured is None:
        if len(candidates) != 1:
            raise RuntimeError(f"{name}: found {len(candidates)} candidates; set the exact path in CELL 0.")
        chosen = candidates[0]
    else:
        chosen = Path(configured).expanduser().resolve()
    missing = [f for f in required if not (chosen / f).exists()]
    if not chosen.is_dir() or missing:
        raise FileNotFoundError(f"{name}={chosen}: missing {missing}. Check Add Input and uploaded folder structure.")
    return chosen
bundle_candidates = [p.parent for p in (p for root in SEARCH_ROOTS for p in root.rglob("bundle.json")) if (p.parent / "run/last.json").is_file()]
RESTORED = None
if MODE == "resume":
    RESTORED = choose(RESTORE_ROOT, bundle_candidates, "RESTORE_ROOT",
                      ["project", "run/config.json", "run/last.json", "run/best.json", "cache/index.json", "cache/provenance.json", "manifests/videos.json", "manifests/labels.json"])
    if (RESTORED / "bundle.json").is_file() and json.loads((RESTORED / "bundle.json").read_text())["format"] != "vsl-kaggle-tensorflow-v1":
        raise ValueError("Unsupported resume bundle format")
source_candidates = [p.parent for p in (p for root in SEARCH_ROOTS for p in root.rglob("train.py")) if (p.parent / "configs/labels.json").is_file()]
source_choice = SOURCE_ROOT if SOURCE_ROOT is not None else (str(RESTORED / "project") if RESTORED else None)
SOURCE = choose(source_choice, source_candidates, "SOURCE_ROOT", REQUIRED)
if MODE == "fresh":
    landmark_candidates = [p.parent.parent for p in (p for root in SEARCH_ROOTS for p in root.rglob("index.json")) if len(p.parent.name) == 64]
    LANDMARK = choose(LANDMARK_ROOT, landmark_candidates, "LANDMARK_ROOT", [])
    if len(list(LANDMARK.glob("*/index.json"))) != 1:
        raise ValueError("LANDMARK_ROOT must contain exactly one recipe index")
else:
    LANDMARK = RESTORED / "cache"
labels = json.loads((SOURCE / "configs/labels.json").read_text(encoding="utf-8-sig"))
assert len(labels) == 100 and sorted(labels.values()) == list(range(100)), "Invalid local label mapping"
print("Landmark archive:", LANDMARK)
print("Mapping SHA256:", hashlib.sha256((SOURCE / "configs/labels.json").read_bytes()).hexdigest())

In [ ]:
# 4. Copy source into working. Never write to Input or copy the raw RGB dataset.
SESSION = WORKING / SESSION_NAME
PROJECT = SESSION / "project"
SESSION.mkdir(parents=True, exist_ok=True)
def file_hash(p):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(2**20), b""):
            h.update(chunk)
    return h.hexdigest()
def copy_once(src, dst):
    if dst.exists():
        for p in src.rglob("*"):
            if p.is_file() and "__pycache__" not in p.parts:
                q = dst / p.relative_to(src)
                if not q.is_file() or file_hash(p) != file_hash(q):
                    raise RuntimeError(f"Destination differs from source: {q}. Use a new SESSION_NAME.")
    else:
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
PROJECT.mkdir(parents=True, exist_ok=True)
for name in ("vsl", "vsl_tf", "vendor", "configs", "scripts", "docs", "tests", "notebooks"):
    if (SOURCE / name).is_dir():
        copy_once(SOURCE / name, PROJECT / name)
for p in SOURCE.iterdir():
    if p.is_file() and (p.suffix in {".py", ".md", ".txt", ".ini"} or p.name == ".gitignore"):
        dst = PROJECT / p.name
        if dst.exists() and file_hash(dst) != file_hash(p):
            raise RuntimeError(f"Source changed: {dst}; use a new SESSION_NAME")
        if not dst.exists():
            shutil.copy2(p, dst)
MANIFEST = SESSION / "manifests"
CACHE = SESSION / "cache"
PREVIOUS_RUN = SESSION / "previous_run"
if RESTORED:
    for source_name, dest in (("manifests", MANIFEST), ("cache", CACHE), ("run", PREVIOUS_RUN)):
        copy_once(RESTORED / source_name, dest)
os.chdir(PROJECT)
ENV = dict(os.environ, CUBLAS_WORKSPACE_CONFIG=":4096:8", PYTHONUNBUFFERED="1",
           PIP_CACHE_DIR=str(SESSION / "pip_cache"), MPLCONFIGDIR=str(SESSION / "mpl_cache"),
           XDG_CACHE_HOME=str(SESSION / "tool_cache"))
os.environ["MPLCONFIGDIR"] = ENV["MPLCONFIGDIR"]
print("Source:", PROJECT, "Landmark read-only:", LANDMARK, "Outputs:", SESSION)

In [ ]:
# 5. Use Kaggle's existing Python; check GPU/core packages before installing unmet extras.
def cli(*args):
    command = [sys.executable, *map(str, args)]
    print("RUN:", " ".join(command), flush=True)
    subprocess.run(command, cwd=PROJECT, env=ENV, check=True)
REPORT = SESSION / "environment_versions.json"
install_args = ["scripts/kaggle_dependencies.py", "--cache-only", "--requirements", "requirements.txt",
                "--report", REPORT]
if WHEELHOUSE:
    if not Path(WHEELHOUSE).is_dir():
        raise FileNotFoundError("WHEELHOUSE missing; select an actual Input directory")
    install_args += ["--wheelhouse", WHEELHOUSE]
cli(*install_args)
# Pipeline commands use the same Kaggle interpreter in fresh subprocesses.
versions = {k.lower(): v for k, v in json.loads(REPORT.read_text(encoding="utf-8")).items()}
for package in ('tensorflow', 'keras', 'numpy', 'mediapipe', 'opencv-contrib-python', 'pandas', 'scikit-learn', 'matplotlib'):
    print(package, versions.get(package))


In [ ]:
# 6. Runtime config. Preserve splits/mapping/model; all generated artifacts go to working.
base_config = PREVIOUS_RUN / "config.json" if RESTORED else PROJECT / "configs/landmark_train.json"
cfg = json.loads(base_config.read_text(encoding="utf-8-sig"))
RUN_DIR = SESSION / "run"
cfg.update(data_root=str(LANDMARK), manifest_dir=str(MANIFEST), cache_dir=str(CACHE),
           label_mapping=str(PROJECT / "configs/labels.json"), run_dir=str(RUN_DIR))
if RESTORED and cfg["device"] != "cuda":
    raise ValueError("GPU resume requires a TensorFlow GPU checkpoint; no automatic CPU-to-GPU run migration.")
cfg["device"] = "cuda"
for key, value in (("batch_size", BATCH_SIZE), ("workers", NUM_WORKERS), ("amp", AMP), ("epochs", TOTAL_EPOCHS)):
    if value is not None:
        if RESTORED and key == "epochs" and value < cfg["train"][key]:
            raise ValueError("Resume may increase total epochs, not reduce the saved limit")
        if RESTORED and key != "epochs" and cfg["train"][key] != value:
            raise ValueError(f"Resume cannot change train.{key}; use None or the saved value")
        cfg["train"][key] = value
for key, value in (("seed", SEED), ("offline_variants", OFFLINE_VARIANTS)):
    if value is not None:
        if RESTORED and cfg[key] != value:
            raise ValueError(f"Resume cannot change {key}")
        cfg[key] = value
if cfg["rectification"]["enabled"]:
    raise ValueError("This notebook preserves local rectification=false. It does not guess motion table paths.")
CONFIG = SESSION / "runtime_config.json"
if CONFIG.exists() and json.loads(CONFIG.read_text(encoding="utf-8")) != cfg:
    raise RuntimeError("Different runtime config already exists; use a new SESSION_NAME")
CONFIG.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(cfg, indent=2, ensure_ascii=False))
print("Checkpoint every completed epoch; best by val accuracy then val loss. Test is never used for selection.")

## Import existing landmark archive

The importer verifies NPZ checksums, shapes, labels and split counts. It converts 75 ? 4 features to the 54 ? 2 Siformer layout and writes the cache under `/kaggle/working`. Resume uses the bundled cache.


In [ ]:
# 7a. Prepare cache via repo CLI (writes artifacts). Resume NEVER re-extracts landmarks.
if MODE == "fresh":
    if MANIFEST.exists() or CACHE.exists():
        raise RuntimeError("Fresh import destination exists; use a new SESSION_NAME")
    cli("-m", "scripts.import_landmark_cache", "--config", CONFIG, "--source", LANDMARK)
for required in (MANIFEST / "videos.json", MANIFEST / "labels.json", CACHE / "index.json", CACHE / "provenance.json"):
    if not required.is_file():
        raise RuntimeError(f"Missing {required}; incomplete manifest/cache. Do not use a partial cache.")
provenance = json.loads((CACHE / "provenance.json").read_text(encoding="utf-8"))
if not provenance.get("complete"):
    raise RuntimeError("Cache is incomplete")
errors = json.loads((CACHE / "errors.json").read_text(encoding="utf-8"))
print("Cache input errors:", len(errors), "details:", CACHE / "errors.json")
if any(r["split"] in {"train", "validation"} for r in errors):
    raise RuntimeError("Train/validation input errors exist. Inspect errors.json; never silently drop videos.")

In [ ]:
# Read-only integrity/counts/shape preflight; GPU validation forward only, no test forward.
cli('scripts/preflight.py', '--config', CONFIG)
PREFLIGHT_OK = True


## Kiến trúc TensorFlow

Ba encoder độc lập và ProbSparse attention, decoder cross-attention theo implementation gốc. Model summary dưới đây được tạo bởi module dự án.

In [ ]:
cli('scripts/model_summary.py', '--config', CONFIG)


## GPU forward và backward smoke

Preflight ở trên xác nhận forward GPU. Smoke sau chạy một batch train và validation, một optimizer step, lưu vào run riêng. Đây **không** phải kết quả cuối.

In [ ]:
SMOKE_RUN = SESSION / 'smoke_tf'
if MODE == 'fresh':
    if SMOKE_RUN.exists():
        print('Existing smoke run:', SMOKE_RUN)
    else:
        cli('train.py', '--config', CONFIG, '--run-dir', SMOKE_RUN, '--smoke')
else:
    print('Resume mode: previous session already passed smoke; no fresh smoke run.')


## Full training

Dùng đúng entry point `train.py`, checkpoint mỗi epoch; best theo validation Top-1 rồi validation loss. Không dùng test để chọn model.

In [ ]:
if MODE == 'fresh' and RUN_TRAIN:
    if not PREFLIGHT_OK:
        raise RuntimeError('Run preflight first')
    if RUN_DIR.exists():
        raise RuntimeError('Destination run exists; choose a new SESSION_NAME')
    cli('train.py', '--config', CONFIG)
else:
    print('Fresh full training skipped; MODE=', MODE, 'RUN_TRAIN=', RUN_TRAIN)


## Learning curves và best checkpoint

Chỉ hiển thị log thực tế. Có thể chạy lại cell sau khi training bị ngắt ở ranh giới epoch.

In [ ]:
def show_history():
    history_path = RUN_DIR / "history.csv"
    if not history_path.is_file():
        print("History is empty")
        return
    with history_path.open(encoding="utf-8-sig", newline="") as f:
        history = list(csv.DictReader(f))
    if not history:
        print("History is empty")
        return
    import matplotlib.pyplot as plt
    epochs = [int(r["epoch"]) + 1 for r in history]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, metric in zip(axes.flat[:3], ("loss", "accuracy", "macro_f1")):
        for split in ("train", "validation"):
            ax.plot(epochs, [float(r[split + "_" + metric]) for r in history], label=split)
        ax.set(xlabel="Epoch", ylabel=metric)
        ax.legend()
    axes.flat[3].plot(epochs, [float(r["lr"]) for r in history])
    axes.flat[3].set(xlabel="Epoch", ylabel="Learning rate")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "learning_curves.png", dpi=150)
    plt.show()
    for name in ("best.json", "last.json"):
        path = RUN_DIR / name
        if path.exists():
            print(path, json.loads(path.read_text(encoding="utf-8")))
show_history()
print('Best:', RUN_DIR / 'best.json', 'Last:', RUN_DIR / 'last.json')


## Resume phiên sau — mặc định tắt

Upload bundle phiên trước dưới dạng Kaggle Dataset/Model, Add Input, đặt `MODE='resume'`, `RESTORE_ROOT`, `SESSION_NAME` mới và `RUN_RESUME=True` ở cell cấu hình. Dữ liệu RGB vẫn phải Add Input riêng.

In [ ]:
if MODE == "resume" and RUN_RESUME:
    if not globals().get("PREFLIGHT_OK", False):
        raise RuntimeError("Complete preflight before resume")
    if RUN_DIR.exists():
        raise RuntimeError("Destination run already exists; use a new SESSION_NAME")
    cli("train.py", "--config", CONFIG, "--resume", PREVIOUS_RUN / "last.json")
    show_history()
else:
    print("Resume is OFF. Requires MODE='resume', valid RESTORE_ROOT, and RUN_RESUME=True.")

## Test cuối — mặc định tắt

Chỉ bật sau khi đã khóa lựa chọn trên validation. `calibrate.py` khóa best và (nếu chọn) patience từ validation; `evaluate.py` đánh giá mỗi video test một lần.

In [ ]:
if RUN_TEST:
    if not (RUN_DIR / "best.json").is_file():
        raise FileNotFoundError("No validation-selected best checkpoint for this run")
    POLICY = RUN_DIR / "policy.json"
    TEST_DIR = RUN_DIR / "test_v1"
    if POLICY.exists() or TEST_DIR.exists():
        raise RuntimeError("Policy/test output already exists. Do not overwrite or repeat test for selection.")
    args = ["calibrate.py", "--config", CONFIG, "--checkpoint", RUN_DIR / "best.json", "--output", POLICY]
    if ADAPTIVE_INFERENCE:
        args.append("--adaptive")
    cli(*args)
    cli("evaluate.py", "--policy", POLICY, "--output", TEST_DIR)
    print((TEST_DIR / "metrics.json").read_text(encoding="utf-8"))
    print("Report:", TEST_DIR / "classification_report.csv", "Predictions:", TEST_DIR / "predictions.csv")
else:
    print("Test is OFF. Enable RUN_TEST in CELL 0 only after all validation-based decisions are final.")

## Kết quả và hình

Đọc từ metrics/report/confusion thật sau test. Không có kết quả giả.

In [ ]:
if RUN_TEST and (RUN_DIR / 'test_v1/metrics.json').is_file():
    from IPython.display import display, Image
    result = RUN_DIR / 'test_v1'
    print((result / 'metrics.json').read_text(encoding='utf-8'))
    display(Image(filename=str(result / 'confusion_normalized.png')))
else:
    print('No test results yet.')


## Đóng gói output

Bundle chứa source, mapping, manifest, cache và best/last TensorFlow checkpoints. Download từ Kaggle Output; giải nén trước khi Add Input để resume.

In [ ]:
if MAKE_ARCHIVE:
    if not (RUN_DIR / "last.json").is_file():
        print("No completed epoch checkpoint yet; cannot build a resume bundle.")
    else:
        ARCHIVE = WORKING / (SESSION_NAME + "_resume_bundle.zip")
        cli("scripts/kaggle_bundle.py", "--config", CONFIG, "--output", ARCHIVE)
        from IPython.display import display, FileLink
        display(FileLink(str(ARCHIVE)))
        print("Download from Kaggle Output:", ARCHIVE)